# Notebook 3: Building a Knowledge Graph from PDFs

In this notebook, you'll learn how to:
1. Build a knowledge graph directly from **PDF files**
2. Use **automatic schema extraction** (let the LLM figure out the schema)
3. Process a real research paper ("Attention Is All You Need")

## How PDF Processing Works

When you set `from_pdf=True` (the default), the pipeline adds a PDF loading step:

```
PDF file → Extract text → Split into chunks → LLM extracts entities → Write to Neo4j
```

The pipeline uses `pypdf` under the hood to extract text from each page.

## Step 1: Setup

In [16]:
import os
import asyncio
from pathlib import Path
import neo4j
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = neo4j.GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j!")

Connected to Neo4j!


In [17]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

llm = OpenAILLM(
    model_name="gpt-4.1-2025-04-14",
    model_params={
        "temperature": 0,
        "response_format": {"type": "json_object"},
    },
)

embedder = OpenAIEmbeddings(model="text-embedding-3-large")
print("LLM and Embedder ready!")

LLM and Embedder ready!


## Step 2: Check Our PDF Files

We have two research papers in the `data/` folder:
- **Attention Is All You Need** (Vaswani et al., 2017) — the Transformer paper
- **BERT** (Devlin et al., 2018) — the BERT paper

In [18]:
from pypdf import PdfReader

data_dir = Path("../data")

# Check available PDFs
pdf_files = list(data_dir.glob("*.pdf"))
for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)
    print(f"{pdf_file.name}: {len(reader.pages)} pages")

Patent - 2021 - Systems and methods for production and display of dynamically linked slide presentations.pdf: 21 pages
Patent - 2016 - Querying medical claims data.pdf: 28 pages
Patent - 2023 - Entity matching with machine learning fuzzy logic.pdf: 18 pages
Patent - 2022 - Central user interface for accessing and upgrading of dataset integrations.pdf: 28 pages
Patent - 2023 - Synchronization and tagging of image and text data.pdf: 24 pages
patent - 2014 - Overview user interface of emergency call data of a law enforcement agency.pdf: 39 pages
Patent - 2018 - Systems, methods, user interfaces and algorithms for performing database analysis and search of information involving structured and or semi-structured data.pdf: 47 pages
Patent - 2017 - Interactive geospatial map.pdf: 71 pages
Patent - 2022 - Controlling user actions and access to electronic data assets.pdf: 110 pages
Patent - 2022 - Systems and methods for data replication synchronization.pdf: 20 pages
Patent - 2022 - Code list b

## Step 3: Build KG from a PDF

Let's start with the Transformer paper. You can choose to define a manual schema or let the LLM figure one out automatically.

In [19]:
# ── Toggle: set to False to let the LLM generate its own schema ──
USE_MANUAL_SCHEMA = False

NODE_TYPES = [
    "Person",
    {
        "label": "Organization",
        "description": "A company, university, or research lab",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Model",
        "description": "A machine learning model or architecture",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Technique",
        "description": "A method, algorithm, or technique",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Dataset",
        "description": "A dataset or benchmark used for evaluation",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Metric",
        "description": "An evaluation metric (e.g., BLEU, perplexity)",
        "properties": [{"name": "name", "type": "STRING"}],
    },
]

RELATIONSHIP_TYPES = [
    "AUTHORED_BY",
    "AFFILIATED_WITH",
    "USES_TECHNIQUE",
    "EVALUATED_ON",
    "ACHIEVES_SCORE_ON",
    "COMPARED_TO",
    {"label": "IMPROVES_UPON", "description": "One model/technique outperforms another"},
]

PATTERNS = [
    ("Model", "AUTHORED_BY", "Person"),
    ("Person", "AFFILIATED_WITH", "Organization"),
    ("Model", "USES_TECHNIQUE", "Technique"),
    ("Model", "EVALUATED_ON", "Dataset"),
    ("Model", "ACHIEVES_SCORE_ON", "Metric"),
    ("Model", "COMPARED_TO", "Model"),
    ("Model", "IMPROVES_UPON", "Model"),
]

if USE_MANUAL_SCHEMA:
    schema = {
        "node_types": NODE_TYPES,
        "relationship_types": RELATIONSHIP_TYPES,
        "patterns": PATTERNS,
    }
    print(f"Using MANUAL schema: {len(NODE_TYPES)} node types, {len(RELATIONSHIP_TYPES)} relationship types")
else:
    schema = "EXTRACTED"
    print("Using AUTO schema: the LLM will extract a schema from the PDF")

Using AUTO schema: the LLM will extract a schema from the PDF


In [ ]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

kg_builder_pdf = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    schema=schema,
    from_pdf=True,
    neo4j_database=NEO4J_DATABASE,
)

# Process all PDFs in the data folder
for pdf_path in sorted(data_dir.glob("*.pdf")):
    print(f"Processing: {pdf_path.name}...")
    result = await kg_builder_pdf.run_async(
        file_path=str(pdf_path),
        document_metadata={"source": "arxiv", "paper": pdf_path.stem},
    )
    print(f"  Done! {result.result}\n")

print("All PDFs processed!")

Processing: Patent - 2010 - Dynamic indexing.pdf...
  Done! {'resolver': {'number_of_nodes_to_resolve': 0, 'number_of_created_nodes': None}}

Processing: Patent - 2011 - Creating data in a data store using a dynamic ontology.pdf...


LLM response has improper format for chunk_index=5


## Step 4: Explore What Was Extracted

In [ ]:
# Summary of the graph
records, _, _ = driver.execute_query(
    """
    MATCH (n)
    WITH labels(n) AS node_labels, count(n) AS count
    RETURN node_labels, count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== All Node Types ===")
for record in records:
    print(f"  {record['node_labels']}: {record['count']}")

In [ ]:
# See the extracted entities (not Chunk/Document nodes)
records, _, _ = driver.execute_query(
    """
    MATCH (n)
    WHERE NOT n:Chunk AND NOT n:Document AND n.name IS NOT NULL
    RETURN labels(n)[0] AS type, n.name AS name
    ORDER BY type, name
    """,
    database_=NEO4J_DATABASE,
)

print("=== Extracted Entities ===")
current_type = None
for record in records:
    if record['type'] != current_type:
        current_type = record['type']
        print(f"\n  [{current_type}]")
    print(f"    - {record['name']}")

In [ ]:
# See extracted relationships
records, _, _ = driver.execute_query(
    """
    MATCH (a)-[r]->(b)
    WHERE NOT a:Chunk AND NOT a:Document AND NOT b:Chunk AND NOT b:Document
    RETURN labels(a)[0] AS from_type, a.name AS from_name,
           type(r) AS relationship,
           labels(b)[0] AS to_type, b.name AS to_name
    ORDER BY relationship, from_name
    """,
    database_=NEO4J_DATABASE,
)

print("=== Knowledge Graph Triples ===")
for record in records:
    print(
        f"  ({record['from_type']}: {record['from_name']}) "
        f"-[{record['relationship']}]-> "
        f"({record['to_type']}: {record['to_name']})"
    )

## Step 5: See the Combined Graph

Both papers are now in the same graph. Let's see what was extracted across all documents.

In [ ]:
# Entity types across all papers
records, _, _ = driver.execute_query(
    """
    MATCH (n)
    WHERE NOT n:Chunk AND NOT n:Document AND n.name IS NOT NULL
    WITH labels(n) AS types, count(n) AS count
    RETURN types, count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== Entity Types Across All Papers ===")
total = 0
for record in records:
    print(f"  {record['types']}: {record['count']}")
    total += record['count']
print(f"\n  Total entities: {total}")

In [ ]:
# Let's see the combined graph now (both papers)
records, _, _ = driver.execute_query(
    """
    MATCH (n)
    WHERE NOT n:Chunk AND NOT n:Document AND n.name IS NOT NULL
    WITH labels(n) AS types, count(n) AS count
    RETURN types, count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== Combined Graph Entity Types ===")
total = 0
for record in records:
    print(f"  {record['types']}: {record['count']}")
    total += record['count']
print(f"\n  Total entities: {total}")

## Step 6: Query Across Papers

Since all papers are in the same graph, we can query across them!

In [ ]:
# Find entities that appear in both papers (connected to both Document nodes)
records, _, _ = driver.execute_query(
    """
    MATCH (d:Document)
    RETURN d.path AS document, d.source AS source
    """,
    database_=NEO4J_DATABASE,
)

print("=== Documents in the Graph ===")
for record in records:
    print(f"  {record['document']} (source: {record['source']})")

In [ ]:
# Find connections between entities from both papers
records, _, _ = driver.execute_query(
    """
    MATCH (a)-[r]->(b)
    WHERE NOT a:Chunk AND NOT a:Document AND NOT b:Chunk AND NOT b:Document
    RETURN labels(a)[0] AS from_type, a.name AS from_name,
           type(r) AS rel, b.name AS to_name
    ORDER BY from_name
    LIMIT 25
    """,
    database_=NEO4J_DATABASE,
)

print("=== Sample Relationships (first 25) ===")
for record in records:
    print(f"  {record['from_name']} -[{record['rel']}]-> {record['to_name']}")

## Summary

In this notebook you learned how to:
- Process PDF files with `SimpleKGPipeline` using `from_pdf=True`
- Define schemas for research papers
- Use automatic schema extraction (just omit the `schema` parameter)
- Build a combined graph from multiple papers

**Key takeaway:** The pipeline handles PDF text extraction automatically. You can process entire papers and let the LLM extract structured knowledge.

**Next up:** [Notebook 4 — Building a Knowledge Graph from Web Pages](./04_kg_from_web.ipynb)

In [ ]:
driver.close()